# Ideal Minimal-Task Hyperparameter Tuning and Benchmarking Notebook

This notebook implements the **Black-Scholes European call minimal task** for the deep hedging project.

It includes:

- no-hedge, Black-Scholes delta, and discrete-time MSE-optimal hedge benchmarks;
- shared Markov MLP architectures with raw and normalized inputs;
- time-separated MLP architectures closest to the project diagram;
- polynomial / linear least-squares hedge baselines;
- train/validation/test splitting;
- premium, P&L, left-tail, turnover, and delta-surface diagnostics;
- saved configs, metrics, histories, plots, and models.

Start with `SMOKE_TEST = True`. Once the notebook runs end-to-end, set `SMOKE_TEST = False` for the full architecture comparison.

---

## Original script overview

Ideal Minimal-Task Hyperparameter Tuning and Benchmarking Script
=
Deep hedging a vanilla European call under Black-Scholes dynamics.

Purpose

This script is designed for a rigorous minimal-task analysis in the MFE
neural-network hedging project. It improves on a basic tuning script by:

1. Using separate train / validation / test sets.
2. Selecting hyperparameters by validation loss, not test RMSE.
3. Evaluating no hedge, sampled Black-Scholes delta, and discrete-time
   mean-square-optimal hedge benchmarks on the same test paths.
4. Comparing architecture families: shared Markov MLP, time-separated MLP,
   and a polynomial/linear least-squares hedge baseline.
5. Comparing raw vs normalized Markov inputs.
6. Comparing output constraints: sigmoid, linear, scaled tanh.
7. Recording pricing, hedging, left-tail, turnover, and delta-surface metrics.
7. Saving all configs, histories, metrics, plots, and selected-model weights.
8. Supporting Google Drive persistence in Colab and local fallback.
9. Supporting smoke-test mode before an overnight run.

Main mathematical setup
--
For r = 0, the terminal self-financing hedge portfolio is

    V_N = pi + sum_{n=0}^{N-1} delta_n (S_{n+1} - S_n).

The seller's terminal hedging error is

    HE = V_N - max(S_T - K, 0).

The neural network is trained to minimize E[HE^2].

Recommended workflow
------
1. Run with SMOKE_TEST = True.
2. Confirm all files and plots are saved.
3. Set SMOKE_TEST = False for the full overnight run.

Author note
----
This script is intentionally verbose and heavily commented so that the
implementation can be explained clearly in a report appendix and oral exam.

## How to run

1. Open this notebook in Colab.
2. Run all cells with `SMOKE_TEST = True` first.
3. Confirm the run folder contains results, plots and models.
4. Change `SMOKE_TEST = False` in the **Global switches** cell for the full run.
5. Results will save to Google Drive if mounting succeeds; otherwise they save to local Colab disk.


## 0. Imports

In [1]:
import os
import json
import csv
import time
import math
import shutil
import random
import traceback
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

import tensorflow as tf
import keras
from keras import layers, callbacks, optimizers

## 1. Global switches

In [2]:
SMOKE_TEST = False          # Set False for overnight run
FORCE_RERUN = False        # If False, skips configs with existing metrics
USE_GOOGLE_DRIVE = True    # In Colab, save to Drive; otherwise local fallback
RUN_BOOTSTRAP = True
RUN_DELTA_SURFACE = True
RUN_ROBUSTNESS_GRIDS = False  # Expensive; enable after core tuning works

GLOBAL_SEED = 2026

## 2. Reproducibility

In [3]:
def set_all_seeds(seed: int) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_all_seeds(GLOBAL_SEED)

## 3. Google Drive / output directory setup

In [4]:
def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def make_run_directory() -> Path:
    """Create a timestamped results directory.

    In Colab, attempts to mount Google Drive. If Drive mount fails, falls back
    to /content/mfe_deep_hedging_runs so that the script still runs.
    """
    timestamp = time.strftime("%Y%m%d_%H%M%S")

    if USE_GOOGLE_DRIVE and running_in_colab():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/mfe_deep_hedging_runs")
        except Exception as e:
            print("WARNING: Google Drive mount failed. Falling back to local disk.")
            print("Drive error:", repr(e))
            root = Path("/content/mfe_deep_hedging_runs")
    else:
        root = Path("./mfe_deep_hedging_runs")

    run_dir = root / f"minimal_task_tuning_{timestamp}"
    subdirs = [
        "config",
        "logs",
        "results",
        "plots",
        "plots/delta_surfaces",
        "plots/learning_curves",
        "plots/pnl",
        "models",
        "histories",
        "arrays",
        "robustness",
    ]
    for sub in subdirs:
        (run_dir / sub).mkdir(parents=True, exist_ok=True)
    return run_dir


RUN_DIR = make_run_directory()
print(f"Run directory: {RUN_DIR}")


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


def append_log(message: str) -> None:
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {message}"
    print(line)
    with open(RUN_DIR / "logs" / "run_log.txt", "a", encoding="utf-8") as f:
        f.write(line + "\n")


def save_environment_info() -> None:
    info = {
        "python": os.sys.version,
        "tensorflow": tf.__version__,
        "keras": keras.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "smoke_test": SMOKE_TEST,
        "global_seed": GLOBAL_SEED,
        "run_dir": str(RUN_DIR),
    }
    save_json(info, RUN_DIR / "config" / "environment.json")
    with open(RUN_DIR / "config" / "environment.txt", "w", encoding="utf-8") as f:
        for k, v in info.items():
            f.write(f"{k}: {v}\n")


save_environment_info()

Mounted at /content/drive
Run directory: /content/drive/MyDrive/mfe_deep_hedging_runs/minimal_task_tuning_20260616_172433


## 4. Experiment configuration

In [5]:
@dataclass
class GlobalConfig:
    # Black-Scholes minimal task parameters
    r: float = 0.0
    sigma: float = 0.4
    S0: float = 1.0
    K: float = 0.9
    T: float = 0.5
    N: int = 125

    # Data sizes
    M_train: int = 100_000
    M_val: int = 20_000
    M_test: int = 50_000

    # Training defaults
    max_epochs: int = 60
    early_stopping_patience: int = 8
    reduce_lr_patience: int = 4
    bootstrap_resamples: int = 200

    # Seeds
    train_seed: int = 42
    val_seed: int = 99
    test_seed: int = 123


if SMOKE_TEST:
    CFG = GlobalConfig(
        N=20,
        M_train=2_000,
        M_val=1_000,
        M_test=2_000,
        max_epochs=4,
        early_stopping_patience=2,
        reduce_lr_patience=1,
        bootstrap_resamples=20,
    )
else:
    CFG = GlobalConfig()

CFG.dt = CFG.T / CFG.N  # type: ignore[attr-defined]
save_json(asdict(CFG), RUN_DIR / "config" / "global_config.json")

## 5. Black-Scholes formulas and simulation

In [6]:
def bs_call_price(S: np.ndarray | float, K: float, tau: np.ndarray | float,
                  r: float, sigma: float) -> np.ndarray | float:
    """Black-Scholes European call price. Vectorized in S and tau."""
    S_arr = np.asarray(S, dtype=float)
    tau_arr = np.asarray(tau, dtype=float)
    out = np.maximum(S_arr - K, 0.0)
    mask = tau_arr > 1e-12
    if np.isscalar(tau) and float(tau) <= 1e-12:
        return float(out)

    tau_safe = np.maximum(tau_arr, 1e-12)
    d1 = (np.log(np.maximum(S_arr, 1e-12) / K) + (r + 0.5 * sigma**2) * tau_safe) / (sigma * np.sqrt(tau_safe))
    d2 = d1 - sigma * np.sqrt(tau_safe)
    price = S_arr * norm.cdf(d1) - K * np.exp(-r * tau_safe) * norm.cdf(d2)
    return np.where(mask, price, out)


def bs_call_delta(S: np.ndarray | float, K: float, tau: np.ndarray | float,
                  r: float, sigma: float) -> np.ndarray | float:
    """Black-Scholes European call delta. Vectorized."""
    S_arr = np.asarray(S, dtype=float)
    tau_safe = np.maximum(np.asarray(tau, dtype=float), 1e-12)
    d1 = (np.log(np.maximum(S_arr, 1e-12) / K) + (r + 0.5 * sigma**2) * tau_safe) / (sigma * np.sqrt(tau_safe))
    return norm.cdf(d1)


def discrete_time_optimal_call_delta(S: np.ndarray | float, K: float, tau: np.ndarray | float,
                                     h: float, sigma: float) -> np.ndarray | float:
    """Analytic discrete-time MSE-optimal hedge for a BS call when r = 0.

    This is the finite-grid regression coefficient over [t, t+h]:

        delta_DT(t,s) = E[Phi(S_T)(S_{t+h}-s)|S_t=s]
                        / E[(S_{t+h}-s)^2|S_t=s]

    for a European call payoff under Black-Scholes dynamics with r=0.

    Formula:
        numerator = s^2 exp(sigma^2 h) N(d1 + sigma h / sqrt(tau))
                    - K s N(d2 + sigma h / sqrt(tau))
                    - s C_BS(t,s)
        denominator = s^2 (exp(sigma^2 h)-1)

    where tau = T - t, d1 and d2 are the usual BS terms at (t,s,tau).

    As h -> 0, this converges to the usual Black-Scholes delta.
    """
    s = np.asarray(S, dtype=float)
    tau_arr = np.maximum(np.asarray(tau, dtype=float), h)  # final trading interval has tau=h
    s_safe = np.maximum(s, 1e-12)

    sqrt_tau = np.sqrt(tau_arr)
    d1 = (np.log(s_safe / K) + 0.5 * sigma**2 * tau_arr) / (sigma * sqrt_tau)
    d2 = d1 - sigma * sqrt_tau
    shift = sigma * h / sqrt_tau

    C = bs_call_price(s_safe, K, tau_arr, r=0.0, sigma=sigma)
    numerator = (
        s_safe**2 * np.exp(sigma**2 * h) * norm.cdf(d1 + shift)
        - K * s_safe * norm.cdf(d2 + shift)
        - s_safe * C
    )
    denominator = s_safe**2 * (np.exp(sigma**2 * h) - 1.0)
    return numerator / np.maximum(denominator, 1e-18)


def simulate_gbm(M: int, N: int, S0: float, r: float, sigma: float, T: float,
                 seed: Optional[int] = None) -> np.ndarray:
    """Exact log-Euler simulation of GBM paths. Returns shape (M, N+1)."""
    rng = np.random.default_rng(seed)
    dt = T / N
    Z = rng.standard_normal((M, N)).astype(np.float32)
    log_inc = np.float32((r - 0.5 * sigma**2) * dt) + np.float32(sigma * np.sqrt(dt)) * Z
    log_S = np.float32(np.log(S0)) + np.cumsum(log_inc, axis=1)
    S = np.hstack([np.full((M, 1), S0, dtype=np.float32), np.exp(log_S).astype(np.float32)])
    return S


C0_BS = float(bs_call_price(CFG.S0, CFG.K, CFG.T, CFG.r, CFG.sigma))
append_log(f"Black-Scholes call price C0 = {C0_BS:.8f}")

[2026-06-16 17:24:54] Black-Scholes call price C0 = 0.16411068


## 6. Dataset generation and validation

In [7]:
append_log("Generating train / validation / test paths...")
S_train = simulate_gbm(CFG.M_train, CFG.N, CFG.S0, CFG.r, CFG.sigma, CFG.T, CFG.train_seed)
S_val = simulate_gbm(CFG.M_val, CFG.N, CFG.S0, CFG.r, CFG.sigma, CFG.T, CFG.val_seed)
S_test = simulate_gbm(CFG.M_test, CFG.N, CFG.S0, CFG.r, CFG.sigma, CFG.T, CFG.test_seed)

zeros_train = np.zeros((CFG.M_train, 1), dtype=np.float32)
zeros_val = np.zeros((CFG.M_val, 1), dtype=np.float32)
zeros_test = np.zeros((CFG.M_test, 1), dtype=np.float32)

append_log(f"Train paths: {S_train.shape}, Val paths: {S_val.shape}, Test paths: {S_test.shape}")


def validate_simulation(S: np.ndarray) -> Dict[str, float]:
    ST = S[:, -1]
    log_ST = np.log(ST / CFG.S0)
    return {
        "mean_ST": float(np.mean(ST)),
        "theoretical_mean_ST": float(CFG.S0 * np.exp(CFG.r * CFG.T)),
        "var_log_return": float(np.var(log_ST)),
        "theoretical_var_log_return": float(CFG.sigma**2 * CFG.T),
        "mc_call_price": float(np.mean(np.maximum(ST - CFG.K, 0.0))),
        "bs_call_price": C0_BS,
    }


sim_validation = validate_simulation(S_test)
save_json(sim_validation, RUN_DIR / "results" / "simulation_validation.json")
append_log(f"Simulation validation: {sim_validation}")

[2026-06-16 17:24:54] Generating train / validation / test paths...
[2026-06-16 17:24:55] Train paths: (100000, 126), Val paths: (20000, 126), Test paths: (50000, 126)
[2026-06-16 17:24:55] Simulation validation: {'mean_ST': 1.000923991203308, 'theoretical_mean_ST': 1.0, 'var_log_return': 0.07961791753768921, 'theoretical_var_log_return': 0.08000000000000002, 'mc_call_price': 0.1646336317062378, 'bs_call_price': 0.16411067988848604}


## 7. Metrics and benchmark portfolio functions

In [8]:
def payoff_call(S_paths: np.ndarray, K: float) -> np.ndarray:
    return np.maximum(S_paths[:, -1] - K, 0.0)


def compute_metrics(hedge_error: np.ndarray,
                    deltas: Optional[np.ndarray] = None,
                    S_paths: Optional[np.ndarray] = None,
                    premium: Optional[float] = None,
                    reference_price: Optional[float] = None,
                    runtime_seconds: Optional[float] = None,
                    n_params: Optional[int] = None,
                    best_val_loss: Optional[float] = None,
                    epochs_trained: Optional[int] = None) -> Dict[str, float]:
    he = np.asarray(hedge_error, dtype=float)
    loss = -he
    out: Dict[str, float] = {
        "mean_HE": float(np.mean(he)),
        "std_HE": float(np.std(he)),
        "rmse_HE": float(np.sqrt(np.mean(he**2))),
        "mae_HE": float(np.mean(np.abs(he))),
        "median_HE": float(np.median(he)),
        "q01_HE": float(np.percentile(he, 1)),
        "q05_HE": float(np.percentile(he, 5)),
        "q95_HE": float(np.percentile(he, 95)),
        "q99_HE": float(np.percentile(he, 99)),
        "min_HE": float(np.min(he)),
        "max_HE": float(np.max(he)),
    }
    centered = he - np.mean(he)
    out.update({
        "std_centered_HE": float(np.std(centered)),
        "rmse_centered_HE": float(np.sqrt(np.mean(centered**2))),
        "mae_centered_HE": float(np.mean(np.abs(centered))),
    })

    for q in [95, 99]:
        var = float(np.percentile(loss, q))
        cvar = float(np.mean(loss[loss >= var])) if np.any(loss >= var) else var
        out[f"loss_VaR_{q}"] = var
        out[f"loss_CVaR_{q}"] = cvar

    if deltas is not None:
        d = np.asarray(deltas, dtype=float)
        out["mean_abs_delta"] = float(np.mean(np.abs(d)))
        out["max_abs_delta"] = float(np.max(np.abs(d)))
        if S_paths is not None:
            prev = np.zeros((d.shape[0], 1))
            d_changes = np.diff(np.concatenate([prev, d], axis=1), axis=1)
            dollar_turnover = np.sum(np.abs(d_changes) * S_paths[:, :d.shape[1]], axis=1)
            share_turnover = np.sum(np.abs(d_changes), axis=1)
            out["mean_dollar_turnover"] = float(np.mean(dollar_turnover))
            out["median_dollar_turnover"] = float(np.median(dollar_turnover))
            out["mean_share_turnover"] = float(np.mean(share_turnover))

    if premium is not None:
        out["premium"] = float(premium)
    if reference_price is not None:
        out["reference_price"] = float(reference_price)
        if premium is not None:
            out["premium_error"] = float(premium - reference_price)
            out["abs_premium_error"] = float(abs(premium - reference_price))
    if runtime_seconds is not None:
        out["runtime_seconds"] = float(runtime_seconds)
    if n_params is not None:
        out["n_params"] = float(n_params)
    if best_val_loss is not None:
        out["best_val_loss"] = float(best_val_loss)
    if epochs_trained is not None:
        out["epochs_trained"] = float(epochs_trained)
    return out


def benchmark_no_hedge(S_paths: np.ndarray, premium: float) -> Tuple[np.ndarray, np.ndarray]:
    payoff = payoff_call(S_paths, CFG.K)
    he = premium - payoff
    deltas = np.zeros((S_paths.shape[0], CFG.N), dtype=np.float32)
    return he, deltas


def benchmark_bs_delta(S_paths: np.ndarray, premium: float) -> Tuple[np.ndarray, np.ndarray]:
    M, steps = S_paths.shape
    N = steps - 1
    dt = CFG.T / N
    tau = CFG.T - np.arange(N) * dt
    S_steps = S_paths[:, :N]
    deltas = bs_call_delta(S_steps, CFG.K, tau.reshape(1, -1), CFG.r, CFG.sigma).astype(np.float32)
    pnl = np.sum(deltas * (S_paths[:, 1:] - S_paths[:, :N]), axis=1)
    he = premium + pnl - payoff_call(S_paths, CFG.K)
    return he, deltas


def benchmark_dt_optimal_delta(S_paths: np.ndarray, premium: float) -> Tuple[np.ndarray, np.ndarray]:
    M, steps = S_paths.shape
    N = steps - 1
    dt = CFG.T / N
    tau = CFG.T - np.arange(N) * dt
    S_steps = S_paths[:, :N]
    deltas = discrete_time_optimal_call_delta(S_steps, CFG.K, tau.reshape(1, -1), dt, CFG.sigma).astype(np.float32)
    pnl = np.sum(deltas * (S_paths[:, 1:] - S_paths[:, :N]), axis=1)
    he = premium + pnl - payoff_call(S_paths, CFG.K)
    return he, deltas

## 8. Neural network hedger model

In [9]:
class SharedMLPHedger(keras.Model):
    """Weight-sharing Markov MLP hedge model for the minimal task.

    The model learns a scalar premium pi and a delta function

        delta_n = f_theta(features(S_n, tau_n)).

    The same delta network is reused across all time steps.
    """

    def __init__(self,
                 N: int,
                 K: float,
                 T: float,
                 dt: float,
                 n_units: int,
                 n_layers: int,
                 activation: str,
                 feature_mode: str,
                 output_mode: str,
                 delta_scale: float = 2.0,
                 initial_premium: float = 0.0,
                 l2_reg: float = 0.0,
                 dropout: float = 0.0,
                 name: str = "shared_mlp_hedger"):
        super().__init__(name=name)
        self.N = int(N)
        self.K = float(K)
        self.T = float(T)
        self.dt = float(dt)
        self.feature_mode = feature_mode
        self.output_mode = output_mode
        self.delta_scale = float(delta_scale)
        self.dropout = float(dropout)

        self.tau_vals = tf.constant((T - np.arange(N) * dt).astype(np.float32), dtype=tf.float32)
        self.pi = self.add_weight(
            name="premium",
            shape=(),
            initializer=keras.initializers.Constant(float(initial_premium)),
            trainable=True,
            dtype=tf.float32,
        )

        reg = keras.regularizers.l2(l2_reg) if l2_reg > 0 else None
        self.hidden_layers = []
        for i in range(n_layers):
            self.hidden_layers.append(layers.Dense(n_units, activation=activation, kernel_regularizer=reg, name=f"dense_{i}"))
            if dropout > 0:
                self.hidden_layers.append(layers.Dropout(dropout, name=f"dropout_{i}"))
        self.out_layer = layers.Dense(1, activation="linear", kernel_regularizer=reg, name="delta_raw")

    def _features(self, S_steps: tf.Tensor) -> tf.Tensor:
        """Build feature tensor of shape (batch, N, d)."""
        tau_2d = tf.ones_like(S_steps) * self.tau_vals[tf.newaxis, :]
        eps = tf.constant(1e-7, dtype=tf.float32)
        if self.feature_mode == "raw":
            return tf.stack([S_steps, tau_2d], axis=-1)
        if self.feature_mode == "normalized":
            log_m = tf.math.log(tf.maximum(S_steps, eps) / self.K)
            tau_scaled = tau_2d / self.T
            return tf.stack([log_m, tau_scaled], axis=-1)
        if self.feature_mode == "normalized_plus_price":
            log_m = tf.math.log(tf.maximum(S_steps, eps) / self.K)
            tau_scaled = tau_2d / self.T
            rel_price = S_steps  # S0=1 in baseline; keep explicit for simplicity
            return tf.stack([log_m, tau_scaled, rel_price], axis=-1)
        raise ValueError(f"Unknown feature_mode: {self.feature_mode}")

    def delta_from_features(self, feat_2d: tf.Tensor, training: bool = False) -> tf.Tensor:
        x = feat_2d
        for layer in self.hidden_layers:
            x = layer(x, training=training)
        raw = self.out_layer(x, training=training)
        if self.output_mode == "sigmoid":
            return tf.sigmoid(raw)
        if self.output_mode == "linear":
            return raw
        if self.output_mode == "scaled_tanh":
            return self.delta_scale * tf.tanh(raw)
        raise ValueError(f"Unknown output_mode: {self.output_mode}")

    def compute_deltas(self, S_path: tf.Tensor, training: bool = False) -> tf.Tensor:
        S_steps = S_path[:, :self.N]
        feat_3d = self._features(S_steps)
        feat_2d = tf.reshape(feat_3d, (-1, feat_3d.shape[-1]))
        delta_flat = self.delta_from_features(feat_2d, training=training)
        deltas = tf.reshape(delta_flat, (-1, self.N))
        return deltas

    def call(self, S_path: tf.Tensor, training: bool = False) -> tf.Tensor:
        S_path = tf.cast(S_path, tf.float32)
        deltas = self.compute_deltas(S_path, training=training)
        dS = S_path[:, 1:] - S_path[:, :self.N]
        pnl = tf.reduce_sum(deltas * dS, axis=1, keepdims=True)
        payoff = tf.maximum(S_path[:, self.N:self.N+1] - self.K, 0.0)
        hedge_err = self.pi + pnl - payoff
        return hedge_err


class TimeSeparatedMLPHedger(keras.Model):
    """Time-separated MLP hedge model.

    This architecture is closest to the project diagram: each rebalancing date
    has its own small neural network,

        delta_n = f_{theta,n}(features(S_n, tau_n)).

    It is more flexible than a shared Markov MLP but has many more parameters
    and does not impose smoothness across time. Under Black-Scholes, the true
    hedge is a single smooth Markov function, so this architecture is mainly a
    useful baseline for assessing whether weight sharing is beneficial.
    """

    def __init__(self,
                 N: int,
                 K: float,
                 T: float,
                 dt: float,
                 n_units: int,
                 n_layers: int,
                 activation: str,
                 feature_mode: str,
                 output_mode: str,
                 delta_scale: float = 2.0,
                 initial_premium: float = 0.0,
                 l2_reg: float = 0.0,
                 dropout: float = 0.0,
                 name: str = "time_separated_mlp_hedger"):
        super().__init__(name=name)
        self.N = int(N)
        self.K = float(K)
        self.T = float(T)
        self.dt = float(dt)
        self.feature_mode = feature_mode
        self.output_mode = output_mode
        self.delta_scale = float(delta_scale)
        self.dropout = float(dropout)

        self.tau_vals = tf.constant((T - np.arange(N) * dt).astype(np.float32), dtype=tf.float32)
        self.pi = self.add_weight(
            name="premium",
            shape=(),
            initializer=keras.initializers.Constant(float(initial_premium)),
            trainable=True,
            dtype=tf.float32,
        )

        reg = keras.regularizers.l2(l2_reg) if l2_reg > 0 else None
        self.step_hidden_layers: List[List[keras.layers.Layer]] = []
        self.step_out_layers: List[keras.layers.Layer] = []
        for n in range(N):
            hidden: List[keras.layers.Layer] = []
            for i in range(n_layers):
                hidden.append(layers.Dense(n_units, activation=activation, kernel_regularizer=reg, name=f"step{n}_dense{i}"))
                if dropout > 0:
                    hidden.append(layers.Dropout(dropout, name=f"step{n}_dropout{i}"))
            self.step_hidden_layers.append(hidden)
            self.step_out_layers.append(layers.Dense(1, activation="linear", kernel_regularizer=reg, name=f"step{n}_delta_raw"))

    def _features(self, S_steps: tf.Tensor) -> tf.Tensor:
        """Build feature tensor of shape (batch, N, d)."""
        tau_2d = tf.ones_like(S_steps) * self.tau_vals[tf.newaxis, :]
        eps = tf.constant(1e-7, dtype=tf.float32)
        if self.feature_mode == "raw":
            return tf.stack([S_steps, tau_2d], axis=-1)
        if self.feature_mode == "normalized":
            log_m = tf.math.log(tf.maximum(S_steps, eps) / self.K)
            tau_scaled = tau_2d / self.T
            return tf.stack([log_m, tau_scaled], axis=-1)
        if self.feature_mode == "normalized_plus_price":
            log_m = tf.math.log(tf.maximum(S_steps, eps) / self.K)
            tau_scaled = tau_2d / self.T
            return tf.stack([log_m, tau_scaled, S_steps], axis=-1)
        raise ValueError(f"Unknown feature_mode: {self.feature_mode}")

    def _apply_output_constraint(self, raw: tf.Tensor) -> tf.Tensor:
        if self.output_mode == "sigmoid":
            return tf.sigmoid(raw)
        if self.output_mode == "linear":
            return raw
        if self.output_mode == "scaled_tanh":
            return self.delta_scale * tf.tanh(raw)
        raise ValueError(f"Unknown output_mode: {self.output_mode}")

    def compute_deltas(self, S_path: tf.Tensor, training: bool = False) -> tf.Tensor:
        S_path = tf.cast(S_path, tf.float32)
        S_steps = S_path[:, :self.N]
        feat_3d = self._features(S_steps)
        outputs = []
        for n in range(self.N):
            x = feat_3d[:, n, :]
            for layer in self.step_hidden_layers[n]:
                x = layer(x, training=training)
            raw = self.step_out_layers[n](x, training=training)
            outputs.append(self._apply_output_constraint(raw))
        return tf.concat(outputs, axis=1)

    def delta_on_values(self, S_values: np.ndarray, tau_values: np.ndarray) -> np.ndarray:
        """Evaluate the time-separated hedge at arbitrary (S, tau) pairs.

        Since this model has one network per grid time, arbitrary tau values are
        mapped to the nearest rebalancing index. This makes delta-surface plots
        meaningful while making clear that the architecture itself is discrete
        in time.
        """
        S_values = np.asarray(S_values, dtype=np.float32).reshape(-1)
        tau_values = np.asarray(tau_values, dtype=np.float32).reshape(-1)
        n_idx = np.rint((self.T - tau_values) / self.dt).astype(int)
        n_idx = np.clip(n_idx, 0, self.N - 1)

        out = np.empty_like(S_values, dtype=np.float32)
        for n in np.unique(n_idx):
            mask = n_idx == n
            s = tf.convert_to_tensor(S_values[mask].reshape(-1, 1), dtype=tf.float32)
            tau = tf.ones_like(s) * tf.constant(self.T - n * self.dt, dtype=tf.float32)
            eps = tf.constant(1e-7, dtype=tf.float32)
            if self.feature_mode == "raw":
                x = tf.concat([s, tau], axis=1)
            elif self.feature_mode == "normalized":
                x = tf.concat([tf.math.log(tf.maximum(s, eps) / self.K), tau / self.T], axis=1)
            elif self.feature_mode == "normalized_plus_price":
                x = tf.concat([tf.math.log(tf.maximum(s, eps) / self.K), tau / self.T, s], axis=1)
            else:
                raise ValueError(self.feature_mode)
            for layer in self.step_hidden_layers[n]:
                x = layer(x, training=False)
            raw = self.step_out_layers[n](x, training=False)
            out[mask] = self._apply_output_constraint(raw).numpy().reshape(-1)
        return out

    def call(self, S_path: tf.Tensor, training: bool = False) -> tf.Tensor:
        S_path = tf.cast(S_path, tf.float32)
        deltas = self.compute_deltas(S_path, training=training)
        dS = S_path[:, 1:] - S_path[:, :self.N]
        pnl = tf.reduce_sum(deltas * dS, axis=1, keepdims=True)
        payoff = tf.maximum(S_path[:, self.N:self.N+1] - self.K, 0.0)
        hedge_err = self.pi + pnl - payoff
        return hedge_err


def make_hedger_model(cfg: Dict[str, Any]) -> keras.Model:
    """Factory for NN architecture families."""
    architecture = cfg.get("architecture", "shared_mlp")
    common_kwargs = dict(
        N=CFG.N,
        K=CFG.K,
        T=CFG.T,
        dt=CFG.dt,  # type: ignore[attr-defined]
        n_units=cfg["n_units"],
        n_layers=cfg["n_layers"],
        activation=cfg["activation"],
        feature_mode=cfg["feature_mode"],
        output_mode=cfg["output_mode"],
        delta_scale=cfg["delta_scale"],
        initial_premium=C0_BS,
        l2_reg=cfg["l2_reg"],
        dropout=cfg["dropout"],
        name=f"hedger_{cfg['label']}",
    )
    if architecture == "shared_mlp":
        return SharedMLPHedger(**common_kwargs)
    if architecture == "time_separated_mlp":
        return TimeSeparatedMLPHedger(**common_kwargs)
    raise ValueError(f"Unknown architecture: {architecture}")


def mse_hedge_loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
    del y_true
    return tf.reduce_mean(tf.square(y_pred))

## 9. Candidate hyperparameter grid

In [10]:
def make_candidate(label: str, architecture: str, feature_mode: str, n_units: int,
                   n_layers: int, activation: str, output_mode: str,
                   learning_rate: float = 1e-3, batch_size: int = 256,
                   dropout: float = 0.0, l2_reg: float = 0.0,
                   delta_scale: float = 2.0) -> Dict[str, Any]:
    """Small helper to keep the grid consistent across architecture families."""
    return {
        "label": label,
        "architecture": architecture,
        "feature_mode": feature_mode,
        "n_units": n_units,
        "n_layers": n_layers,
        "activation": activation,
        "output_mode": output_mode,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "dropout": dropout,
        "l2_reg": l2_reg,
        "delta_scale": delta_scale,
    }


def build_candidate_grid() -> List[Dict[str, Any]]:
    """Create a compact but meaningful architecture and hyperparameter grid.

    In full mode, this grid compares:
    - shared Markov MLPs: delta_n = f_theta(S_n, tau_n)
    - time-separated MLPs: delta_n = f_{theta,n}(S_n), closest to the project diagram
    - raw vs normalized financial features
    - sigmoid vs linear vs scaled tanh outputs
    - width/depth/activation/learning-rate/batch-size sensitivity

    The polynomial/linear hedge is trained separately below because it is not a
    neural network and has a closed-form least-squares fit.
    """
    if SMOKE_TEST:
        return [
            make_candidate("shared_raw_16u_1L_tanh_sigmoid", "shared_mlp", "raw", 16, 1, "tanh", "sigmoid"),
            make_candidate("shared_norm_16u_1L_tanh_sigmoid", "shared_mlp", "normalized", 16, 1, "tanh", "sigmoid"),
            make_candidate("time_sep_norm_8u_1L_tanh_sigmoid", "time_separated_mlp", "normalized", 8, 1, "tanh", "sigmoid"),
        ]

    grid: List[Dict[str, Any]] = []

    # Core capacity sweep for the theoretically preferred shared Markov MLP.
    for n_units, n_layers, act in [
        (16, 1, "tanh"),
        (16, 2, "tanh"),
        (32, 1, "tanh"),
        (32, 2, "tanh"),
        (32, 3, "tanh"),
        (64, 2, "tanh"),
        (64, 3, "tanh"),
        (128, 2, "tanh"),
        (64, 2, "elu"),
        (64, 2, "relu"),
        (64, 2, "softplus"),
    ]:
        grid.append(make_candidate(
            f"shared_norm_{n_units}u_{n_layers}L_{act}_sigmoid",
            "shared_mlp", "normalized", n_units, n_layers, act, "sigmoid",
        ))

    # Raw-input comparison, matching the earlier script's basic architecture.
    for n_units, n_layers in [(32, 2), (64, 2)]:
        grid.append(make_candidate(
            f"shared_raw_{n_units}u_{n_layers}L_tanh_sigmoid",
            "shared_mlp", "raw", n_units, n_layers, "tanh", "sigmoid",
        ))

    # Time-separated MLPs: closer to the original project diagram, but less
    # parameter-efficient because each rebalancing date has its own subnetwork.
    # Keep this grid small so it is a controlled architecture comparison rather
    # than an expensive second tuning project.
    for n_units, n_layers in [(8, 1), (16, 1), (16, 2), (32, 1)]:
        grid.append(make_candidate(
            f"time_sep_norm_{n_units}u_{n_layers}L_tanh_sigmoid",
            "time_separated_mlp", "normalized", n_units, n_layers, "tanh", "sigmoid",
        ))

    # Output constraint comparison on the leading shared architecture.
    for output_mode in ["linear", "scaled_tanh"]:
        grid.append(make_candidate(
            f"shared_norm_64u_2L_tanh_{output_mode}",
            "shared_mlp", "normalized", 64, 2, "tanh", output_mode,
        ))

    # Learning-rate and batch-size sensitivity for the leading shared style.
    for lr in [5e-4, 1e-4]:
        grid.append(make_candidate(
            f"shared_norm_64u_2L_tanh_sigmoid_lr{lr:g}",
            "shared_mlp", "normalized", 64, 2, "tanh", "sigmoid", learning_rate=lr,
        ))
    for bs in [512, 1024]:
        grid.append(make_candidate(
            f"shared_norm_64u_2L_tanh_sigmoid_bs{bs}",
            "shared_mlp", "normalized", 64, 2, "tanh", "sigmoid", batch_size=bs,
        ))

    return grid


CANDIDATES = build_candidate_grid()
pd.DataFrame(CANDIDATES).to_csv(RUN_DIR / "config" / "candidate_grid.csv", index=False)
append_log(f"Number of NN candidates: {len(CANDIDATES)}")

[2026-06-16 17:24:55] Number of NN candidates: 23


## 10. Training and evaluation helpers

In [11]:
def train_one_candidate(cfg: Dict[str, Any], candidate_index: int) -> Tuple[keras.Model, Dict[str, Any], pd.DataFrame]:
    """Train one NN candidate and return model, metrics, history."""
    label = cfg["label"]
    candidate_dir = RUN_DIR / "models" / label
    candidate_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = candidate_dir / "metrics.json"
    save_json(cfg, candidate_dir / "config.json")

    if metrics_path.exists() and not FORCE_RERUN:
        append_log(f"Skipping {label}; metrics already exist.")
        # Re-training is skipped, but this branch is intentionally simple.
        # In practice, rerun if you need the model object loaded.

    keras.backend.clear_session()
    set_all_seeds(GLOBAL_SEED + 10_000 + candidate_index)

    model = make_hedger_model(cfg)
    # Build model variables by calling once.
    _ = model(tf.convert_to_tensor(S_train[:2]), training=False)

    model.compile(optimizer=optimizers.Adam(learning_rate=cfg["learning_rate"]), loss=mse_hedge_loss)

    best_weights_path = candidate_dir / "best.weights.h5"
    cb_list = [
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=CFG.reduce_lr_patience,
            min_lr=1e-5,
            verbose=0,
        ),
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=CFG.early_stopping_patience,
            restore_best_weights=True,
            verbose=0,
        ),
        callbacks.ModelCheckpoint(
            filepath=str(best_weights_path),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),
    ]

    append_log(f"Training candidate {candidate_index+1}/{len(CANDIDATES)}: {label}")
    t0 = time.time()
    hist = model.fit(
        S_train,
        zeros_train,
        validation_data=(S_val, zeros_val),
        epochs=CFG.max_epochs,
        batch_size=cfg["batch_size"],
        callbacks=cb_list,
        verbose=1 if SMOKE_TEST else 0,
    )
    runtime = time.time() - t0

    history_df = pd.DataFrame(hist.history)
    history_df.to_csv(candidate_dir / "history.csv", index=False)
    history_df.to_csv(RUN_DIR / "histories" / f"{label}_history.csv", index=False)

    # Evaluate on validation and test separately.
    val_errors = model.predict(S_val, verbose=0)[:, 0]
    test_errors = model.predict(S_test, verbose=0)[:, 0]
    test_deltas = model.compute_deltas(tf.convert_to_tensor(S_test), training=False).numpy()
    pi_nn = float(model.pi.numpy())

    val_rmse = float(np.sqrt(np.mean(val_errors**2)))
    best_val_loss = float(np.min(hist.history["val_loss"]))
    epochs_trained = len(hist.history["loss"])
    n_params = int(model.count_params() + 1)  # add scalar premium if not counted by summary in some versions

    metrics = compute_metrics(
        test_errors,
        deltas=test_deltas,
        S_paths=S_test,
        premium=pi_nn,
        reference_price=C0_BS,
        runtime_seconds=runtime,
        n_params=n_params,
        best_val_loss=best_val_loss,
        epochs_trained=epochs_trained,
    )
    metrics.update({
        "label": label,
        "architecture": cfg.get("architecture", "shared_mlp"),
        "feature_mode": cfg["feature_mode"],
        "output_mode": cfg["output_mode"],
        "n_units": cfg["n_units"],
        "n_layers": cfg["n_layers"],
        "activation": cfg["activation"],
        "learning_rate": cfg["learning_rate"],
        "batch_size": cfg["batch_size"],
        "dropout": cfg["dropout"],
        "l2_reg": cfg["l2_reg"],
        "delta_scale": cfg["delta_scale"],
        "val_rmse_HE": val_rmse,
        "selection_metric": best_val_loss,
    })

    save_json(metrics, metrics_path)
    pd.DataFrame([metrics]).to_csv(candidate_dir / "metrics.csv", index=False)
    model.save_weights(candidate_dir / "final.weights.h5")

    # Save arrays for selected analysis, but not every huge object.
    np.savez_compressed(
        RUN_DIR / "arrays" / f"{label}_test_arrays.npz",
        hedge_error=test_errors.astype(np.float32),
        deltas=test_deltas[: min(2000, len(test_deltas))].astype(np.float32),
        S_test=S_test[: min(2000, len(S_test))].astype(np.float32),
        payoff=payoff_call(S_test, CFG.K).astype(np.float32),
    )

    append_log(
        f"Completed {label}: val_loss={best_val_loss:.6g}, "
        f"test_RMSE={metrics['rmse_HE']:.6g}, pi={pi_nn:.6g}, time={runtime:.1f}s"
    )
    return model, metrics, history_df

## 11. Run benchmarks

In [12]:
def evaluate_benchmarks() -> pd.DataFrame:
    rows = []
    for name, func in [
        ("no_hedge", benchmark_no_hedge),
        ("bs_delta_sampled", benchmark_bs_delta),
        ("discrete_time_mse_optimal", benchmark_dt_optimal_delta),
    ]:
        he, dlt = func(S_test, C0_BS)
        m = compute_metrics(he, deltas=dlt, S_paths=S_test, premium=C0_BS, reference_price=C0_BS)
        m.update({"label": name, "strategy_type": "benchmark"})
        rows.append(m)
        np.savez_compressed(RUN_DIR / "arrays" / f"benchmark_{name}.npz",
                            hedge_error=he.astype(np.float32),
                            deltas=dlt[: min(2000, len(dlt))].astype(np.float32))
    df = pd.DataFrame(rows)
    df.to_csv(RUN_DIR / "results" / "benchmark_metrics.csv", index=False)
    save_json(rows, RUN_DIR / "results" / "benchmark_metrics.json")
    return df

## 11b. Polynomial / linear least-squares hedge baseline

In [13]:
def polynomial_delta_features(S_paths: np.ndarray, degree: int) -> np.ndarray:
    """Feature tensor for a non-NN polynomial hedge.

    The hedge has the interpretable form

        delta_n = beta_0 + beta_1 x_n + beta_2 u_n + ...,

    where x_n = log(S_n/K) and u_n = tau_n/T. The coefficients are fitted by
    least squares to the terminal payoff through the self-financing portfolio.

    Returns shape (M, N, p).
    """
    S_steps = S_paths[:, :CFG.N]
    tau = (CFG.T - np.arange(CFG.N) * CFG.dt).reshape(1, -1)  # type: ignore[attr-defined]
    x = np.log(np.maximum(S_steps, 1e-12) / CFG.K)
    u = np.ones_like(S_steps) * (tau / CFG.T)

    feats = [np.ones_like(x), x, u]
    if degree >= 2:
        feats.extend([x**2, x*u, u**2])
    if degree >= 3:
        feats.extend([x**3, (x**2)*u, x*(u**2), u**3])
    return np.stack(feats, axis=-1).astype(np.float64)


def polynomial_design_matrix(S_paths: np.ndarray, degree: int) -> np.ndarray:
    """Build terminal portfolio design matrix.

    For each path m,

        V_N^{(m)} = pi + sum_k beta_k sum_n phi_k(S_n,tau_n) Delta S_n.

    This is linear in [pi, beta_1, ..., beta_p], so it can be fitted by least
    squares against the payoff.
    """
    feats = polynomial_delta_features(S_paths, degree)
    dS = (S_paths[:, 1:] - S_paths[:, :CFG.N]).astype(np.float64)
    hedge_cols = np.sum(feats * dS[:, :, None], axis=1)
    return np.concatenate([np.ones((S_paths.shape[0], 1)), hedge_cols], axis=1)


def fit_polynomial_hedge(degree: int, alphas: List[float]) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Fit polynomial hedge coefficients and select ridge alpha by validation RMSE."""
    X_train = polynomial_design_matrix(S_train, degree)
    y_train = payoff_call(S_train, CFG.K).astype(np.float64)
    X_val = polynomial_design_matrix(S_val, degree)
    y_val = payoff_call(S_val, CFG.K).astype(np.float64)

    best_beta: Optional[np.ndarray] = None
    best_info: Dict[str, Any] = {}
    for alpha in alphas:
        reg = np.eye(X_train.shape[1]) * alpha
        reg[0, 0] = 0.0  # do not penalize premium/intercept
        beta = np.linalg.solve(X_train.T @ X_train + reg, X_train.T @ y_train)
        val_he = X_val @ beta - y_val
        val_rmse = float(np.sqrt(np.mean(val_he**2)))
        if best_beta is None or val_rmse < best_info["val_rmse_HE"]:
            best_beta = beta
            best_info = {"degree": degree, "ridge_alpha": alpha, "val_rmse_HE": val_rmse}
    assert best_beta is not None
    return best_beta, best_info


def evaluate_polynomial_hedge(degree: int, label: str) -> Dict[str, Any]:
    """Fit and evaluate a polynomial/linear hedge baseline."""
    alphas = [0.0, 1e-10, 1e-8, 1e-6, 1e-4]
    t0 = time.time()
    beta, fit_info = fit_polynomial_hedge(degree, alphas)
    runtime = time.time() - t0

    feats_test = polynomial_delta_features(S_test, degree)
    coeff = beta[1:]
    deltas = np.tensordot(feats_test, coeff, axes=([2], [0])).astype(np.float32)
    pnl = np.sum(deltas * (S_test[:, 1:] - S_test[:, :CFG.N]), axis=1)
    premium = float(beta[0])
    he = premium + pnl - payoff_call(S_test, CFG.K)

    metrics = compute_metrics(
        he,
        deltas=deltas,
        S_paths=S_test,
        premium=premium,
        reference_price=C0_BS,
        runtime_seconds=runtime,
        n_params=len(beta),
        best_val_loss=fit_info["val_rmse_HE"]**2,
        epochs_trained=0,
    )
    metrics.update({
        "label": label,
        "strategy_type": "polynomial_least_squares_baseline",
        "architecture": "polynomial_linear_hedge" if degree == 1 else f"polynomial_degree_{degree}_hedge",
        "feature_mode": "log_moneyness_tau_polynomial",
        "output_mode": "linear_in_features",
        "n_units": 0,
        "n_layers": 0,
        "activation": "closed_form_least_squares",
        "learning_rate": 0.0,
        "batch_size": 0,
        "dropout": 0.0,
        "l2_reg": fit_info["ridge_alpha"],
        "delta_scale": np.nan,
        "val_rmse_HE": fit_info["val_rmse_HE"],
        "selection_metric": fit_info["val_rmse_HE"]**2,
    })

    np.savez_compressed(
        RUN_DIR / "arrays" / f"benchmark_{label}.npz",
        hedge_error=he.astype(np.float32),
        deltas=deltas[: min(2000, len(deltas))].astype(np.float32),
        beta=beta.astype(np.float64),
    )
    save_json(metrics, RUN_DIR / "results" / f"{label}_metrics.json")
    return metrics


def evaluate_polynomial_baselines() -> pd.DataFrame:
    rows = [
        evaluate_polynomial_hedge(1, "linear_polynomial_hedge_deg1"),
        evaluate_polynomial_hedge(2, "quadratic_polynomial_hedge_deg2"),
    ]
    df = pd.DataFrame(rows)
    df.to_csv(RUN_DIR / "results" / "polynomial_baseline_metrics.csv", index=False)
    save_json(rows, RUN_DIR / "results" / "polynomial_baseline_metrics.json")
    return df


benchmark_df = evaluate_benchmarks()
append_log("Benchmarks evaluated.")

polynomial_df = evaluate_polynomial_baselines()
append_log("Polynomial / linear hedge baselines evaluated.")

[2026-06-16 17:24:58] Benchmarks evaluated.
[2026-06-16 17:25:02] Polynomial / linear hedge baselines evaluated.


## 12. Run candidate tuning loop

In [ ]:
all_metrics: List[Dict[str, Any]] = []
all_histories: Dict[str, pd.DataFrame] = {}
trained_models: Dict[str, keras.Model] = {}

for i, cfg in enumerate(CANDIDATES):
    try:
        model, metrics, hist_df = train_one_candidate(cfg, i)
        all_metrics.append(metrics)
        all_histories[cfg["label"]] = hist_df
        trained_models[cfg["label"]] = model
        pd.DataFrame(all_metrics).to_csv(RUN_DIR / "results" / "nn_tuning_metrics_so_far.csv", index=False)
    except Exception as e:
        append_log(f"ERROR in candidate {cfg.get('label')}: {repr(e)}")
        with open(RUN_DIR / "logs" / "errors.txt", "a", encoding="utf-8") as f:
            f.write(f"\n--- {cfg.get('label')} ---\n")
            f.write(traceback.format_exc())
            f.write("\n")

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(RUN_DIR / "results" / "nn_tuning_metrics.csv", index=False)

if len(metrics_df) == 0:
    raise RuntimeError("No NN candidates completed successfully.")

# Select by validation loss only.
best_row = metrics_df.sort_values("selection_metric", ascending=True).iloc[0].to_dict()
best_label = str(best_row["label"])
append_log(f"Best NN selected by validation loss: {best_label}")
save_json(best_row, RUN_DIR / "results" / "selected_model_metrics.json")

# Also save diagnostic best-by-test, but do not use for selection.
best_by_test = metrics_df.sort_values("rmse_HE", ascending=True).iloc[0].to_dict()
save_json(best_by_test, RUN_DIR / "results" / "diagnostic_best_by_test_rmse_DO_NOT_USE_FOR_SELECTION.json")

[2026-06-16 17:25:06] Training candidate 1/23: shared_norm_16u_1L_tanh_sigmoid
[2026-06-16 17:26:10] Completed shared_norm_16u_1L_tanh_sigmoid: val_loss=0.000134651, test_RMSE=0.0115155, pi=0.164008, time=59.7s
[2026-06-16 17:26:10] Training candidate 2/23: shared_norm_16u_2L_tanh_sigmoid
[2026-06-16 17:27:18] Completed shared_norm_16u_2L_tanh_sigmoid: val_loss=6.71941e-05, test_RMSE=0.00817603, pi=0.164053, time=63.0s
[2026-06-16 17:27:18] Training candidate 3/23: shared_norm_32u_1L_tanh_sigmoid
[2026-06-16 17:28:22] Completed shared_norm_32u_1L_tanh_sigmoid: val_loss=0.000100552, test_RMSE=0.00993558, pi=0.16407, time=59.2s
[2026-06-16 17:28:23] Training candidate 4/23: shared_norm_32u_2L_tanh_sigmoid
[2026-06-16 17:29:32] Completed shared_norm_32u_2L_tanh_sigmoid: val_loss=6.60341e-05, test_RMSE=0.00810695, pi=0.164051, time=65.3s
[2026-06-16 17:29:33] Training candidate 5/23: shared_norm_32u_3L_tanh_sigmoid
[2026-06-16 17:30:51] Completed shared_norm_32u_3L_tanh_sigmoid: val_loss=6

## 13. Delta-surface analysis

In [ ]:
def evaluate_model_delta_on_grid(model: keras.Model,
                                 S_grid: np.ndarray,
                                 tau_grid: np.ndarray) -> np.ndarray:
    """Evaluate learned delta f(S,tau) on a grid.

    Shared MLPs are naturally continuous in tau. Time-separated MLPs have one
    network per rebalancing date, so arbitrary tau values are mapped to the
    nearest grid date using the model's delta_on_values method.

    Returns array with shape (len(tau_grid), len(S_grid)).
    """
    SS, TT = np.meshgrid(S_grid, tau_grid)

    if isinstance(model, TimeSeparatedMLPHedger):
        return model.delta_on_values(SS.ravel(), TT.ravel()).reshape(len(tau_grid), len(S_grid))

    if not isinstance(model, SharedMLPHedger):
        raise TypeError(f"Delta surface is implemented for NN hedgers only, got {type(model)}")

    if model.feature_mode == "raw":
        feat = np.stack([SS.ravel(), TT.ravel()], axis=1).astype(np.float32)
    elif model.feature_mode == "normalized":
        feat = np.stack([np.log(np.maximum(SS.ravel(), 1e-7) / CFG.K), TT.ravel() / CFG.T], axis=1).astype(np.float32)
    elif model.feature_mode == "normalized_plus_price":
        feat = np.stack([np.log(np.maximum(SS.ravel(), 1e-7) / CFG.K), TT.ravel() / CFG.T, SS.ravel()], axis=1).astype(np.float32)
    else:
        raise ValueError(model.feature_mode)
    deltas = model.delta_from_features(tf.convert_to_tensor(feat), training=False).numpy().reshape(len(tau_grid), len(S_grid))
    return deltas


def delta_surface_analysis(model: keras.Model, label: str) -> Dict[str, float]:
    S_grid = np.linspace(0.4, 1.8, 120)
    tau_grid = np.linspace(max(CFG.dt, 1e-4), CFG.T, 100)  # avoid tau=0
    SS, TT = np.meshgrid(S_grid, tau_grid)
    nn_delta = evaluate_model_delta_on_grid(model, S_grid, tau_grid)
    bs_delta = bs_call_delta(SS, CFG.K, TT, CFG.r, CFG.sigma)
    dt_delta = discrete_time_optimal_call_delta(SS, CFG.K, TT, CFG.dt, CFG.sigma)

    err_bs = nn_delta - bs_delta
    err_dt = nn_delta - dt_delta

    stats = {
        "label": label,
        "delta_MAE_vs_BS": float(np.mean(np.abs(err_bs))),
        "delta_RMSE_vs_BS": float(np.sqrt(np.mean(err_bs**2))),
        "delta_max_abs_vs_BS": float(np.max(np.abs(err_bs))),
        "delta_MAE_vs_DT": float(np.mean(np.abs(err_dt))),
        "delta_RMSE_vs_DT": float(np.sqrt(np.mean(err_dt**2))),
        "delta_max_abs_vs_DT": float(np.max(np.abs(err_dt))),
    }

    # Near-strike diagnostic region.
    near_strike = np.abs(SS - CFG.K) <= 0.05
    stats["delta_MAE_near_strike_vs_BS"] = float(np.mean(np.abs(err_bs[near_strike])))
    stats["delta_MAE_near_strike_vs_DT"] = float(np.mean(np.abs(err_dt[near_strike])))

    # Save arrays.
    np.savez_compressed(
        RUN_DIR / "arrays" / f"{label}_delta_surface.npz",
        S_grid=S_grid,
        tau_grid=tau_grid,
        nn_delta=nn_delta,
        bs_delta=bs_delta,
        dt_delta=dt_delta,
        err_bs=err_bs,
        err_dt=err_dt,
    )

    # Plot: NN, BS, abs error vs BS, abs error vs DT.
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    panels = [
        (nn_delta, "NN delta"),
        (bs_delta, "Black-Scholes delta"),
        (np.abs(err_bs), "|NN - BS delta|"),
        (np.abs(err_dt), "|NN - discrete-time optimal delta|"),
    ]
    for ax, (data, title) in zip(axes.flat, panels):
        im = ax.imshow(
            data,
            origin="lower",
            aspect="auto",
            extent=[S_grid.min(), S_grid.max(), tau_grid.min(), tau_grid.max()],
        )
        ax.set_title(title)
        ax.set_xlabel("Stock price S")
        ax.set_ylabel("Time to maturity tau")
        fig.colorbar(im, ax=ax, shrink=0.85)
    fig.suptitle(f"Delta surface diagnostics: {label}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(RUN_DIR / "plots" / "delta_surfaces" / f"{label}_delta_surface.png", dpi=160)
    plt.close()
    return stats


if RUN_DELTA_SURFACE:
    delta_stats_rows = []
    # Best selected model.
    best_model = trained_models[best_label]
    delta_stats_rows.append(delta_surface_analysis(best_model, best_label))
    # Also evaluate a few top configurations, if available.
    top_labels = metrics_df.sort_values("selection_metric").head(min(3, len(metrics_df)))["label"].tolist()
    for lbl in top_labels:
        if lbl != best_label and lbl in trained_models:
            delta_stats_rows.append(delta_surface_analysis(trained_models[lbl], lbl))
    delta_df = pd.DataFrame(delta_stats_rows)
    delta_df.to_csv(RUN_DIR / "results" / "delta_surface_metrics.csv", index=False)
    # Merge selected delta metrics into selected row for convenience.
    selected_delta = delta_df[delta_df["label"] == best_label].iloc[0].to_dict()
    save_json(selected_delta, RUN_DIR / "results" / "selected_delta_surface_metrics.json")
else:
    delta_df = pd.DataFrame()

## 14. Bootstrap paired comparisons

In [ ]:
def metric_rmse(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(x**2)))


def metric_cvar_loss(x: np.ndarray, q: float = 95) -> float:
    loss = -x
    var = np.percentile(loss, q)
    return float(np.mean(loss[loss >= var]))


def paired_bootstrap_diff(a: np.ndarray, b: np.ndarray, metric_fn, n_resamples: int,
                          seed: int = 0) -> Tuple[float, float, float]:
    """Bootstrap CI for metric(a)-metric(b), using common path indices."""
    rng = np.random.default_rng(seed)
    a = np.asarray(a)
    b = np.asarray(b)
    n = len(a)
    diffs = np.empty(n_resamples)
    for i in range(n_resamples):
        idx = rng.integers(0, n, size=n)
        diffs[i] = metric_fn(a[idx]) - metric_fn(b[idx])
    point = metric_fn(a) - metric_fn(b)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(point), float(lo), float(hi)


if RUN_BOOTSTRAP:
    append_log("Running paired bootstrap comparisons...")
    # Load benchmark arrays.
    bench_arrays = {}
    for name in ["no_hedge", "bs_delta_sampled", "discrete_time_mse_optimal"]:
        arr = np.load(RUN_DIR / "arrays" / f"benchmark_{name}.npz")
        bench_arrays[name] = arr["hedge_error"]
    nn_arr = np.load(RUN_DIR / "arrays" / f"{best_label}_test_arrays.npz")["hedge_error"]
    arrays = {"selected_nn": nn_arr, **bench_arrays}
    if "polynomial_df" in globals() and len(polynomial_df) > 0:
        best_poly_label = str(polynomial_df.sort_values("selection_metric").iloc[0]["label"])
        poly_path = RUN_DIR / "arrays" / f"benchmark_{best_poly_label}.npz"
        if poly_path.exists():
            arrays["best_polynomial"] = np.load(poly_path)["hedge_error"]

    rows = []
    pairs = [
        ("selected_nn", "no_hedge"),
        ("selected_nn", "bs_delta_sampled"),
        ("selected_nn", "discrete_time_mse_optimal"),
        ("bs_delta_sampled", "discrete_time_mse_optimal"),
    ]
    if "best_polynomial" in arrays:
        pairs.extend([
            ("selected_nn", "best_polynomial"),
            ("best_polynomial", "bs_delta_sampled"),
            ("best_polynomial", "discrete_time_mse_optimal"),
        ])
    metrics_to_compare = {
        "RMSE": metric_rmse,
        "CVaR95_loss": lambda x: metric_cvar_loss(x, 95),
    }
    for A, B in pairs:
        for metric_name, fn in metrics_to_compare.items():
            point, lo, hi = paired_bootstrap_diff(
                arrays[A], arrays[B], fn, CFG.bootstrap_resamples, seed=GLOBAL_SEED + 777
            )
            rows.append({
                "strategy_A": A,
                "strategy_B": B,
                "metric": metric_name,
                "diff_A_minus_B": point,
                "ci_low": lo,
                "ci_high": hi,
                "bootstrap_resamples": CFG.bootstrap_resamples,
            })
    boot_df = pd.DataFrame(rows)
    boot_df.to_csv(RUN_DIR / "results" / "bootstrap_comparisons.csv", index=False)
else:
    boot_df = pd.DataFrame()

## 15. Plotting functions

In [ ]:
def plot_learning_curves(histories: Dict[str, pd.DataFrame], top_labels: List[str]) -> None:
    n = len(top_labels)
    fig, axes = plt.subplots(n, 1, figsize=(8, max(3, 2.5*n)), sharex=False)
    if n == 1:
        axes = [axes]
    for ax, lbl in zip(axes, top_labels):
        hist = histories[lbl]
        ax.plot(hist["loss"], label="Train")
        ax.plot(hist["val_loss"], label="Validation")
        ax.set_title(lbl)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MSE")
        ax.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / "plots" / "learning_curves" / "top_learning_curves.png", dpi=160)
    plt.close()


def plot_tuning_summary(metrics: pd.DataFrame) -> None:
    df = metrics.sort_values("selection_metric", ascending=True).copy()
    labels = df["label"].tolist()
    y = np.arange(len(labels))

    fig, axes = plt.subplots(2, 2, figsize=(15, max(8, 0.45*len(labels))))
    fig.suptitle("Minimal-task NN hyperparameter tuning summary", fontweight="bold")

    panels = [
        ("rmse_HE", "Test RMSE", "RMSE"),
        ("std_HE", "P&L standard deviation", "Std HE"),
        ("abs_premium_error", "Absolute premium error", "|pi - C0|"),
        ("runtime_seconds", "Training time", "Seconds"),
    ]
    for ax, (col, title, xlabel) in zip(axes.flat, panels):
        ax.barh(y, df[col].values)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=7)
        ax.invert_yaxis()
        ax.set_xlabel(xlabel)
        ax.set_title(title)
    plt.tight_layout()
    plt.savefig(RUN_DIR / "plots" / "tuning_summary.png", dpi=160)
    plt.close()

    # Capacity plot.
    fig, ax = plt.subplots(figsize=(9, 6))
    for _, row in df.iterrows():
        ax.scatter(row["n_params"], row["rmse_HE"], s=70)
        ax.annotate(str(row["label"]), (row["n_params"], row["rmse_HE"]),
                    textcoords="offset points", xytext=(4, 3), fontsize=7)
    ax.set_xlabel("Trainable parameters")
    ax.set_ylabel("Test RMSE")
    ax.set_title("RMSE vs model capacity")
    plt.tight_layout()
    plt.savefig(RUN_DIR / "plots" / "capacity_vs_rmse.png", dpi=160)
    plt.close()


def plot_pnl_histograms(best_label: str) -> None:
    arrays = {}
    for name in ["no_hedge", "bs_delta_sampled", "discrete_time_mse_optimal"]:
        arrays[name] = np.load(RUN_DIR / "arrays" / f"benchmark_{name}.npz")["hedge_error"]
    # Include the best polynomial/linear baseline to compare neural and non-neural learned hedges.
    if "polynomial_df" in globals() and len(polynomial_df) > 0:
        best_poly_label = str(polynomial_df.sort_values("selection_metric").iloc[0]["label"])
        poly_path = RUN_DIR / "arrays" / f"benchmark_{best_poly_label}.npz"
        if poly_path.exists():
            arrays[best_poly_label] = np.load(poly_path)["hedge_error"]
    arrays["selected_nn"] = np.load(RUN_DIR / "arrays" / f"{best_label}_test_arrays.npz")["hedge_error"]

    fig, ax = plt.subplots(figsize=(10, 6))
    for name, arr in arrays.items():
        ax.hist(arr, bins=80, alpha=0.45, density=True, label=name)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title("Terminal hedging-error distribution")
    ax.set_xlabel("Hedging error HE = V_N - payoff")
    ax.set_ylabel("Density")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / "plots" / "pnl" / "pnl_histogram_all_strategies.png", dpi=160)
    plt.close()

    # Left-tail zoom: use common x-limits from 0.5th to 10th percentile of all errors.
    all_vals = np.concatenate(list(arrays.values()))
    lo = np.percentile(all_vals, 0.5)
    hi = np.percentile(all_vals, 15)
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, arr in arrays.items():
        ax.hist(arr, bins=80, alpha=0.45, density=True, label=name, range=(lo, hi))
    ax.set_title("Left-tail zoom of terminal hedging error")
    ax.set_xlabel("Hedging error HE = V_N - payoff")
    ax.set_ylabel("Density")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / "plots" / "pnl" / "left_tail_zoom.png", dpi=160)
    plt.close()

    # ECDF plot.
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, arr in arrays.items():
        x = np.sort(arr)
        y = np.arange(1, len(x)+1) / len(x)
        ax.plot(x, y, label=name)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title("Empirical CDF of terminal hedging error")
    ax.set_xlabel("Hedging error HE")
    ax.set_ylabel("Empirical CDF")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / "plots" / "pnl" / "pnl_ecdf.png", dpi=160)
    plt.close()


def plot_hedge_error_vs_ST(best_label: str) -> None:
    ST = S_test[:, -1]
    payoff = payoff_call(S_test, CFG.K)
    arrays = {
        "bs_delta_sampled": np.load(RUN_DIR / "arrays" / "benchmark_bs_delta_sampled.npz")["hedge_error"],
        "discrete_time_mse_optimal": np.load(RUN_DIR / "arrays" / "benchmark_discrete_time_mse_optimal.npz")["hedge_error"],
        "selected_nn": np.load(RUN_DIR / "arrays" / f"{best_label}_test_arrays.npz")["hedge_error"],
    }
    rng = np.random.default_rng(GLOBAL_SEED)
    idx = rng.choice(len(ST), size=min(5000, len(ST)), replace=False)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
    for ax, (name, he) in zip(axes, arrays.items()):
        ax.scatter(ST[idx], he[idx], s=5, alpha=0.35)
        ax.axhline(0, color="black", linestyle="--", linewidth=1)
        ax.axvline(CFG.K, color="red", linestyle=":", linewidth=1, label="Strike")
        ax.set_title(name)
        ax.set_xlabel("Terminal stock price S_T")
    axes[0].set_ylabel("Hedging error")
    plt.tight_layout()
    plt.savefig(RUN_DIR / "plots" / "hedge_error_vs_ST.png", dpi=160)
    plt.close()


def plot_pathwise_hedges(best_model: keras.Model, best_label: str) -> None:
    S = S_test
    ST = S[:, -1]
    itm_idx = int(np.where(ST > CFG.K)[0][0])
    otm_idx = int(np.where(ST < CFG.K)[0][0])
    near_idx = int(np.argmin(np.abs(ST - CFG.K)))
    choices = [(itm_idx, "itm"), (otm_idx, "otm"), (near_idx, "near_strike")]

    times = np.linspace(0, CFG.T, CFG.N + 1)
    tau = CFG.T - np.arange(CFG.N) * CFG.dt  # type: ignore[attr-defined]

    for idx, tag in choices:
        s_path = S[idx:idx+1]
        s1d = s_path[0]
        payoff = max(s1d[-1] - CFG.K, 0.0)

        nn_delta = best_model.compute_deltas(tf.convert_to_tensor(s_path), training=False).numpy()[0]
        bs_delta = bs_call_delta(s1d[:CFG.N], CFG.K, tau, CFG.r, CFG.sigma)
        dt_delta = discrete_time_optimal_call_delta(s1d[:CFG.N], CFG.K, tau, CFG.dt, CFG.sigma)  # type: ignore[arg-type]

        def portfolio_from_delta(pi: float, delta: np.ndarray) -> np.ndarray:
            V = np.empty(CFG.N + 1)
            V[0] = pi
            dS = np.diff(s1d)
            for n in range(CFG.N):
                V[n+1] = V[n] + delta[n] * dS[n]
            return V

        nn_V = portfolio_from_delta(float(best_model.pi.numpy()), nn_delta)
        bs_V = portfolio_from_delta(C0_BS, bs_delta)
        dt_V = portfolio_from_delta(C0_BS, dt_delta)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        fig.suptitle(f"Pathwise hedge example: {tag}, S_T={s1d[-1]:.4f}, payoff={payoff:.4f}")
        ax = axes[0]
        ax.plot(times, s1d, label="Stock price")
        ax.plot(times, nn_V, label="NN hedge")
        ax.plot(times, bs_V, label="BS delta hedge")
        ax.plot(times, dt_V, label="DT-optimal hedge")
        ax.axhline(payoff, color="black", linestyle=":", label="Payoff")
        ax.set_xlabel("Time")
        ax.set_ylabel("Value")
        ax.set_title("Portfolio value")
        ax.legend(fontsize=8)

        ax = axes[1]
        ax.plot(times[:-1], nn_delta, label="NN delta")
        ax.plot(times[:-1], bs_delta, label="BS delta")
        ax.plot(times[:-1], dt_delta, label="DT-optimal delta")
        ax.set_xlabel("Time")
        ax.set_ylabel("Delta")
        ax.set_title("Delta path")
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(RUN_DIR / "plots" / f"pathwise_{tag}_{best_label}.png", dpi=160)
        plt.close()


# Make plots.
plt.style.use("seaborn-v0_8-whitegrid")
top_labels = metrics_df.sort_values("selection_metric").head(min(5, len(metrics_df)))["label"].tolist()
plot_learning_curves(all_histories, top_labels)
plot_tuning_summary(metrics_df)
plot_pnl_histograms(best_label)
plot_hedge_error_vs_ST(best_label)
plot_pathwise_hedges(trained_models[best_label], best_label)

## 16. Final summary tables

In [ ]:
def build_final_strategy_table() -> pd.DataFrame:
    rows = []
    # Benchmarks
    bench = pd.read_csv(RUN_DIR / "results" / "benchmark_metrics.csv")
    for _, row in bench.iterrows():
        r = row.to_dict()
        r["strategy"] = r.pop("label")
        rows.append(r)
    # Polynomial / linear baselines
    if "polynomial_df" in globals() and len(polynomial_df) > 0:
        for _, row in polynomial_df.iterrows():
            r = row.to_dict()
            r["strategy"] = r.pop("label")
            rows.append(r)
    # Selected NN
    selected = metrics_df[metrics_df["label"] == best_label].iloc[0].to_dict()
    selected["strategy"] = f"selected_nn__{best_label}"
    rows.append(selected)
    return pd.DataFrame(rows)


final_strategy_df = build_final_strategy_table()
final_strategy_df.to_csv(RUN_DIR / "results" / "minimal_task_strategy_comparison.csv", index=False)

# Compact LaTeX table.
compact_cols = [
    "strategy", "premium", "reference_price", "mean_HE", "std_HE", "rmse_HE",
    "mae_HE", "q01_HE", "q05_HE", "loss_CVaR_95", "mean_dollar_turnover",
]
available_cols = [c for c in compact_cols if c in final_strategy_df.columns]
with open(RUN_DIR / "results" / "minimal_task_strategy_comparison.tex", "w", encoding="utf-8") as f:
    f.write(final_strategy_df[available_cols].to_latex(index=False, float_format="%.6f"))

# Tuning table LaTeX.
# Sort before column-subsetting.
tuning_cols = [
    "label", "architecture", "feature_mode", "output_mode", "n_units", "n_layers", "activation",
    "n_params", "selection_metric", "best_val_loss", "val_rmse_HE", "rmse_HE", "std_HE", "loss_CVaR_95",
    "abs_premium_error", "runtime_seconds",
]

available_tuning_cols = [c for c in tuning_cols if c in metrics_df.columns]

sort_col = "selection_metric" if "selection_metric" in metrics_df.columns else (
    "best_val_loss" if "best_val_loss" in metrics_df.columns else "rmse_HE"
)

tuning_export_df = metrics_df.sort_values(sort_col, ascending=True)[available_tuning_cols]

with open(RUN_DIR / "results" / "nn_tuning_metrics.tex", "w", encoding="utf-8") as f:
    f.write(tuning_export_df.to_latex(index=False, float_format="%.6f"))

# Combined master results.
master_df = pd.concat([
    benchmark_df.assign(group="benchmark"),
    polynomial_df.assign(group="polynomial_linear_baseline"),
    metrics_df.assign(group="nn_candidate"),
], ignore_index=True, sort=False)
master_df.to_csv(RUN_DIR / "results" / "master_results.csv", index=False)
save_json(master_df.to_dict(orient="records"), RUN_DIR / "results" / "master_results.json")

## 17. Optional robustness grids

In [ ]:
def run_robustness_placeholder() -> None:
    """Placeholder note saved for report planning.

    Full robustness grids require retraining the selected architecture for each
    strike/rebalancing scenario. This function records the planned scenarios so
    the experiment can be extended without changing the core tuning pipeline.
    """
    robustness_plan = {
        "moneyness_grid": [0.8, 0.9, 1.0, 1.1, 1.2],
        "rebalancing_steps": [25, 50, 125, 252],
        "recommended_selected_architecture": best_label,
        "note": "Retrain selected architecture for each K or N using the same train/val/test protocol.",
    }
    save_json(robustness_plan, RUN_DIR / "robustness" / "robustness_plan.json")


run_robustness_placeholder()

## 18. Final archive

In [ ]:
append_log("Creating final archive...")
archive_base = str(RUN_DIR / "final_archive")
archive_path = shutil.make_archive(archive_base, "zip", root_dir=RUN_DIR)
append_log(f"Archive created: {archive_path}")

append_log("All done.")
print("\n" + "="*80)
print("Minimal-task tuning and benchmark run complete.")
print(f"Results folder: {RUN_DIR}")
print(f"Archive: {archive_path}")
print(f"Selected model by validation loss: {best_label}")
print("="*80)